# 第7周｜岭惩罚与训练内验证

学习目标：标准化、L2收缩、alpha、交叉验证、过拟合。

__数据：默认使用官方 PRIME-CVD 5 万人数据__（`python run.py download` 下载，来源与 MD5 核验记录在 `data/raw/manifest.json`）。把 `MODE` 改成 `"demo"` 可以切换到离线演示数据，但演示数据是另外生成的，**不是 PRIME-CVD**，结果不能当作官方结果。

本周验收：提交折内Brier结果；解释为何不能在全队列先标准化。

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# 从项目根目录、notebooks目录或其他项目内目录启动均可。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请在解压后的项目目录中启动 Jupyter。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from primecvd.common import load_config, provenance_label
from primecvd.pipeline import prepare
from primecvd.splitting import select_split

MODE = "official"  # 官方数据；离线练习或跑测试时可改为 "demo"（演示数据不是 PRIME-CVD）
cfg = load_config(ROOT / "configs" / f"{MODE}.json")
cfg["output_dir"] = f"outputs/notebooks_{MODE}"
print(provenance_label(cfg))
df, splits = prepare(cfg)
train = select_split(df, splits, "train")
valid = select_split(df, splits, "valid")
# 不创建test DataFrame；模型试验只使用train和valid。
print("队列、训练集、验证集：", len(df), len(train), len(valid))

# 命令行绘图模块采用无窗口后端；在Notebook中显式启用行内图像。
get_ipython().run_line_magic("matplotlib", "inline")
from primecvd import pubplot as pp   # 与精讲篇相同的顶刊作图风格（在启用行内图像之后设置）
pp.setup()

OFFICIAL SOURCE FILES / 官方下载文件
队列、训练集、验证集： 50000 35000 7500


## 惩罚强度只能在训练数据内部选择
每个交叉验证训练折都要重新估计插补值、标准化参数、系数和基线危险。下面完整执行训练集内三折；根据平均IPCW Brier选择alpha，不查看测试集。

In [2]:
from sklearn.model_selection import StratifiedKFold
from primecvd.survival import CoxModel
from primecvd.metrics import brier_ipcw
folds = list(StratifiedKFold(n_splits=cfg["cv_folds"], shuffle=True,
                             random_state=cfg["seed"]+2).split(train, train.cvd_event))
rows = []
for alpha in cfg["ridge_alphas"]:
    for fold, (i_train, i_valid) in enumerate(folds, 1):
        fold_train, fold_valid = train.iloc[i_train], train.iloc[i_valid]
        model = CoxModel(cfg["features"], alpha=alpha).fit(fold_train)
        loss = brier_ipcw(fold_train.cvd_time, fold_train.cvd_event,
                         fold_valid.cvd_time, fold_valid.cvd_event,
                         model.predict_risk(fold_valid, cfg["horizon_years"]),
                         cfg["horizon_years"], cfg["min_censor_survival"])
        rows.append({"alpha":alpha, "fold":fold, "IPCW_Brier":loss})
cv_results = pd.DataFrame(rows)
display(cv_results)

,alpha,fold,IPCW_Brier
0,0.000,1,0.027711
1,0.000,2,0.028256
2,0.000,3,0.027744
3,0.001,1,0.027729
4,0.001,2,0.028251
5,0.001,3,0.027721
6,0.010,1,0.027944
7,0.010,2,0.028313
8,0.010,3,0.027715


In [3]:
mean_loss = cv_results.groupby("alpha").IPCW_Brier.mean().sort_values(kind="stable")
selected_alpha = float(mean_loss.index[0])
display(mean_loss)
print("训练内CV选择的alpha：", selected_alpha)

alpha
0.001    0.027901
0.000    0.027904
0.010    0.027991
Name: IPCW_Brier, dtype: float64

训练内CV选择的alpha： 0.001


alpha=0代表不加惩罚。如果0胜出，就如实记录，不能为了展示“岭回归有效”而强迫选择非零值。惩罚可能降低方差，也可能增加偏差；这个练习的目标是正确比较，不是保证某种算法获胜。

## 独立练习
指出上面代码哪一行保证每折重新拟合预处理。解释为什么先在全队列标准化再交叉验证不规范。不要把普通Cox的p值套用到岭惩罚系数上。

---
本周复盘：请用自己的话记录一个学会的概念、一个发现的错误和一个待解决问题。

特别提醒：不能依据最终测试集选择alpha。